In [1]:
#!/tomcat/python3env/bin/python3
import os,sys,re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import math
import seaborn as sns
from matplotlib.pyplot import figure, show
from scipy.stats import spearmanr, variation,zscore
import io
import cgi
import base64
from collections import OrderedDict
import catheat
from catheat import plot as cplt

wd = r"/Users/yingweihu/project/HN_Lung/Head and Neck & Lung/LSCC"

path1 = os.path.join(wd, "LSCC_proteomics_gene_abundance_log2_reference_intensity_normalized_Tumor.txt")
path2 = os.path.join(wd, "LSCC_RNAseq_gene_RSEM_coding_UQ_1500_log2_Tumor.txt")

omics1=pd.read_csv(path1,sep="\t",header=0,index_col=0)
omics2=pd.read_csv(path2,sep="\t",header=0,index_col=0)



/var/folders/3n/v4wj_tqs36zg_qvs2vvr4_mh0000gn/T/ipykernel_59306/758411458.py:13: DeprecationWarning: 'cgi' is deprecated and slated for removal in Python 3.13
  import cgi


In [2]:
out_dir = os.path.join(wd, "sampleclustering")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)


In [3]:



omics1=omics1.apply (pd.to_numeric, errors='coerce').dropna()
omics2=omics2.apply (pd.to_numeric, errors='coerce').dropna()
omics1 = omics1.loc[~omics1.index.duplicated(keep='first'),~omics1.columns.duplicated(keep='first')]
omics2 = omics2.loc[~omics2.index.duplicated(keep='first'),~omics2.columns.duplicated(keep='first')]
# annotation=pd.read_csv(wo+os.sep+"omics3.txt",sep="\t",header=0,index_col=0)
geneinorder=set(omics1.index).intersection(set(omics2.index))
sampleinorder=set(omics1.columns).intersection(set(omics2.columns))

if len(geneinorder)>9 and len(sampleinorder)>9:
    omics2a=omics2.loc[list(geneinorder),list(sampleinorder)]
    omics1a=omics1.loc[list(geneinorder),list(sampleinorder)]
    omics1a.to_csv(os.path.join(out_dir,"omics1a.txt"),sep="\t")
    omics2a.to_csv(os.path.join(out_dir,"omics2a.txt"),sep="\t")    
    # omics2az=pd.DataFrame(np.power(2,omics2a),columns=omics2a.columns, index=omics2a.index)
    omics2az=pd.DataFrame(omics2a,columns=omics2a.columns, index=omics2a.index)
    omics1az=pd.DataFrame(omics1a,columns=omics1a.columns, index=omics1a.index)

if len(sys.argv)>3:
    annotation=pd.read_csv(os.path.join(out_dir, "LSCC_clinical_data_Tumor.txt"),sep="\t",header=0,index_col=0)
    


In [4]:

import statsmodels.stats.multitest as smt
corr=[]
for index, row in omics1a.iterrows():
    corr1,p=spearmanr(omics1a.loc[index,:].T,omics2a.loc[index,:].T)
    cv1 = variation(omics1a.loc[index,:], axis=0)
    cv2 = variation(omics1a.loc[index,:], axis=0)
    corr.append([index,corr1,p,cv1,cv2])
#     print([index,corr1,p])
df_corrtumorRNAproteinGene=pd.DataFrame(np.asarray(corr))
df_corrtumorRNAproteinGene.apply(pd.to_numeric, errors='ignore', downcast='float')
df_corrtumorRNAproteinGene.columns=["Gene","Tumor Spearman Correlation","Tumor P","CV1","CV2"]
df_corrtumorRNAproteinGene["Tumor Spearman Correlation"]=df_corrtumorRNAproteinGene["Tumor Spearman Correlation"].astype(float)
df_corrtumorRNAproteinGene.set_index('Gene', inplace=True)
df_corrtumorRNAproteinGene['Tumor BH adjusted P'] =  smt.multipletests(df_corrtumorRNAproteinGene['Tumor P'].astype(float), method='fdr_bh')[1]
# # mRNA-protein correlation

if(len(df_corrtumorRNAproteinGene.index)>1000):
    cvhigh=df_corrtumorRNAproteinGene.astype(float).nlargest(1000,'CV1').index
else:
    cvhigh=df_corrtumorRNAproteinGene.astype(float).nlargest(round(len(df_corrtumorRNAproteinGene.index)),'CV1').index
genehigh=df_corrtumorRNAproteinGene.astype(float).nlargest(round(len(df_corrtumorRNAproteinGene.index)/20),'Tumor Spearman Correlation').index
genelow=df_corrtumorRNAproteinGene.astype(float).nsmallest(round(len(df_corrtumorRNAproteinGene.index)/20),'Tumor Spearman Correlation').index
genehigh=set(cvhigh).intersection(set(genehigh))
genelow=set(genelow).intersection(set(cvhigh))


/var/folders/3n/v4wj_tqs36zg_qvs2vvr4_mh0000gn/T/ipykernel_59306/3563924966.py:4: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr1,p=spearmanr(omics1a.loc[index,:].T,omics2a.loc[index,:].T)
/var/folders/3n/v4wj_tqs36zg_qvs2vvr4_mh0000gn/T/ipykernel_59306/3563924966.py:10: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_corrtumorRNAproteinGene.apply(pd.to_numeric, errors='ignore', downcast='float')
/var/folders/3n/v4wj_tqs36zg_qvs2vvr4_mh0000gn/T/ipykernel_59306/3563924966.py:10: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_corrtumorRNAproteinGene.apply(pd.to_numeric, errors='ignore', downcast='float')


In [5]:


from collections import defaultdict

import pandas as pd
from scipy.cluster.hierarchy import dendrogram, set_link_color_palette
from scipy.cluster.hierarchy import linkage
import seaborn as sns
from matplotlib.colors import rgb2hex, colorConverter


def labels2cmap(labels, sample_info, color_sys='hls'):
    import seaborn as sns
    si = sample_info.replace(np.nan, 'N/A')
    cmap = {}
    for label in labels:
        x = si.loc[:, label].unique()
        print(x)
        cmap[label] = dict(zip(x, sns.color_palette(color_sys, len(list(x)))))
    print(cmap)
    return cmap


# def labels2colors(labels,sample_info,key_col='SPL',color_sys='hls'):
#     cmap = labels2cmap(labels,sample_info,color_sys)
#     si = sample_info.loc[:, [key_col] + labels]

def labels2colors(labels, sample_info, key_col='SPL', color_sys='hls'):
    cmap = labels2cmap(labels, sample_info, color_sys)
    si = sample_info.loc[:, [key_col] + labels]
    si = si.set_index(key_col)
    for label in labels:
        si[label] = si[label].map(cmap[label])
    return si
    



def newclustermap2(df,name):
    df=pd.DataFrame(zscore(df,axis=1,ddof=1),columns=df.columns, index=df.index)
    # link = linkage(df.T, metric='correlation', method='ward')
    # Calculate correlation distance matrix and convert to Euclidean
    corr_matrix = np.corrcoef(df.T)
    # Convert correlation to distance (1 - correlation)
    dist_matrix = 1 - corr_matrix
    # Convert to Euclidean distance
    dist_matrix_euclidean = np.sqrt(2 * dist_matrix)
    
    # Use the Euclidean distances with Ward's method
    link = linkage(dist_matrix_euclidean, method='ward')
    
    den = dendrogram(link, labels=df.columns)
    cluster_idxs = defaultdict(list)
    for c, pi in zip(den['color_list'], den['icoord']):
        for leg in pi[1:3]:
            i = (leg - 5.0) / 10.0
            if abs(i - int(i)) < 1e-5:
                cluster_idxs[c].append(int(i))
    
    corr=[]
    # cluster_classes = Clusters()
    for c, l in cluster_idxs.items():
        i_l = [den['ivl'][i] for i in l]
        #cluster_classes[c] = i_l
        for x in range(len(i_l)): 
            corr.append([c,i_l[x]])
            
    corr2=pd.DataFrame(corr) 
    corr2.columns=["group","SPL"]
    flatui = ["g", "r", "c", "m", "y", "k","b"]
    sns.palplot(sns.color_palette(flatui))
    mycolors2 = labels2cmap(["group"], corr2, color_sys=flatui)
    col_colors2a = labels2colors(["group"],corr2,color_sys=flatui)
    col_colors2a =  col_colors2a.loc[~ col_colors2a.index.duplicated(keep='first')]
    sns.clustermap(df, z_score=0,vmax=2,vmin=-2,cmap='RdBu_r',col_colors=col_colors2a, 
                   col_linkage=link,figsize=(5, 5),
                  )
    corr3=corr2
    corr3.set_index('SPL', inplace=True,drop=True)
    corr3=corr3.loc[~ corr3.index.duplicated(keep='first')]
    plt.savefig(os.path.join(out_dir,f"{name}.png"),dpi=100,bbox_inches='tight')
    return corr3



In [6]:
genehigh = sorted(list(genehigh))

In [7]:

mm1=omics1az.loc[genehigh,:]
my_col_colors1=newclustermap2(mm1,"figure3")


['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}
['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}


/Users/yingweihu/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seaborn/matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)


In [10]:

mm2=omics2az.loc[genehigh,:]
my_col_colors2=newclustermap2(mm2,"figure4")


genelow = sorted(list(genelow))

mm3=omics1az.loc[genelow,:]
my_col_colors3=newclustermap2(mm3,"figure5")
mm4=omics2az.loc[genelow,:]
my_col_colors4=newclustermap2(mm4,"figure6")



['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}
['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}


/Users/yingweihu/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seaborn/matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)


['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}
['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}
['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}
['C1' 'C2']
{'group': {'C1': (0.0, 0.5, 0.0), 'C2': (1.0, 0.0, 0.0)}}


In [16]:
# Add suffixes to distinguish columns from different merges
types = pd.merge(my_col_colors1, my_col_colors2, how='left', left_index=True, right_index=True, 
                suffixes=('_1', '_2'))
types = pd.merge(types, my_col_colors3, how='left', left_index=True, right_index=True,
                suffixes=('', '_3'))
types = pd.merge(types, my_col_colors4, how='left', left_index=True, right_index=True,
                suffixes=('', '_4'))
types.columns=["group1","group2","group3","group4"]
if len(sys.argv)>3:
    types = pd.merge(types, annotation.T, how='left', left_index=True, right_index=True)


In [18]:

import catheat
from catheat import plot as cplt

f, ax3 = plt.subplots(figsize=(5,2))
plt.tight_layout()
cm3={'g':"g","r":"r","c":"c","m":"m","y":"y","b":"b","k":"k"}
cplt.heatmap( types.T,legend=False,cmap=cm3,ax=ax3, xticklabels = False,linewidth=0.1,linecolor="black" )
ax3.set_yticklabels(types.columns, rotation=0,fontsize=10)
plt.savefig(os.path.join(out_dir,"figure7.png"),dpi=100,bbox_inches='tight')


/Users/yingweihu/opt/anaconda3/envs/py311/lib/python3.11/site-packages/catheat/plot.py:111: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  numerical_data = data.applymap(mapper)


In [24]:
# First create a mapping of unique values to numbers
unique_values = np.unique(types.values)
value_to_num = {val: i for i, val in enumerate(unique_values)}

# Convert the data to numerical values
types_numeric = types.replace(value_to_num)

# Create custom colormap from the original colors
colors = ['green', 'red', 'cyan', 'magenta', 'yellow', 'blue', 'black']
custom_cmap = matplotlib.colors.ListedColormap(colors[:len(unique_values)])

# Create the plot
f, ax3 = plt.subplots(figsize=(5,2))
plt.tight_layout()

sns.heatmap(types_numeric.T, 
            cmap=custom_cmap,
            cbar=False,
            xticklabels=False,
            linewidths=0.1,
            linecolor='black')

ax3.set_yticklabels(types_numeric.columns, rotation=0, fontsize=10)
plt.savefig(os.path.join(out_dir,"figure7-sns.png"), dpi=100, bbox_inches='tight')

/var/folders/3n/v4wj_tqs36zg_qvs2vvr4_mh0000gn/T/ipykernel_59306/2381992523.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  types_numeric = types.replace(value_to_num)


In [20]:


types.replace({ 'g' : 'green', 'r' : 'red', 'c' : 'cyan','m':'magenta', 'y':'yellow','b':'blue','k':'black'}, inplace=True)
types.to_csv(os.path.join(out_dir,"clustering.txt"),sep="\t")


In [22]:


from sklearn.metrics.cluster import adjusted_rand_score
df_ari = pd.DataFrame(columns=types.columns,index=types.columns)
for each in types.columns:
    for each2 in types.columns:
        df_ari.at[each,each2]=adjusted_rand_score(types[each],types[each2])
df_ari=df_ari.astype(float).round(2)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(df_ari.astype(float),vmax=1,vmin=-1,cmap='RdBu_r', annot=True,
            cbar_kws={'label': 'Adjusted Rand Index'}
                  )
plt.savefig(os.path.join(out_dir, "ARI.png"),dpi=100,bbox_inches='tight')